# Build the `training-wheels` dataset (one-time, internet ON)

Companion to `kaggle_build_vllm_wheels.ipynb`, but for the libraries the **training** notebook needs (`peft`, `trl`, `accelerate`, `bitsandbytes`, `datasets`, `transformers`).

Run this notebook once with internet **on**, then convert the output into a Kaggle Dataset called `training-wheels`. After that, `kaggle_training.ipynb` can run with internet **off** (Phase 1 will install `--no-index` from the attached dataset).

## Notebook settings

- **Accelerator**: any GPU (matters for resolving CUDA-compatible bitsandbytes wheels).
- **Internet**: **ON**.
- **Add Data**: nothing required.

## After Run All

Click **Save Version → Save & Run All**, wait for it to finish, open the run's **Output** tab, and click **New Dataset** on the `training-wheels` folder. Name it `training-wheels`. Then attach it to `kaggle_training.ipynb`.

In [ ]:
import shutil, subprocess, sys
from pathlib import Path

DEST = Path("/kaggle/working/training-wheels")
if DEST.exists():
    shutil.rmtree(DEST)
DEST.mkdir(parents=True)

# Match the floors used in kaggle_training.ipynb Phase 1.
REQS = [
    "transformers>=4.45.0,<5",
    "peft>=0.13.0",
    "trl>=0.11.0",
    "accelerate>=0.34.0",
    "bitsandbytes>=0.43.0",
    "datasets>=3.0.0",
    # Common transitive deps that sometimes need refreshing.
    "sentencepiece",
    "tokenizers",
    "safetensors",
]

subprocess.check_call(
    [
        sys.executable, "-m", "pip", "download",
        *REQS,
        "-d", str(DEST),
        "--prefer-binary",
    ]
)

files = sorted(DEST.glob("*"))
print(f"\nDownloaded {len(files)} files into {DEST}:")
for f in files:
    print(f"  {f.name}  ({f.stat().st_size/1e6:.1f} MB)")

## Smoke-test the offline install (optional)

In [ ]:
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install",
        "--no-index",
        "--find-links", str(DEST),
        "transformers", "peft", "trl", "accelerate", "bitsandbytes", "datasets",
    ]
)
import transformers, peft, trl, accelerate, bitsandbytes, datasets
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("datasets:", datasets.__version__)